# Milestone 2 - Linear probe x CLIP ViT-B/16 on PACS

One of the six Milestone 2 run notebooks (one per adaptation method x backbone). They are independent: different people can run different notebooks at the same time. `M2_results.ipynb` builds the final table.

- `lp` on `clip`.

- 4 PACS leave-one-domain-out splits (art, cartoon, photo, sketch held out) with seed 0, i.e. **the same data split as Milestone 1** = **4 runs**.
- **Estimated time** (A100): ~9 min per run, ~0.6 h in total, plus ~10 min of setup per session.
- Model selection: **training-domain validation set**, computed with DomainBed's own code (`model_selection.IIDAccuracySelectionMethod`, the method `collect_results` uses); section 8 also prints the official `collect_results` tables.
- **PACS test_env mapping** (folders sorted alphabetically): `0=art_painting(A)`, `1=cartoon(C)`, `2=photo(P)`, `3=sketch(S)`.

**Saving results, logs and checkpoints** (course guideline: sessions disconnect, save regularly):
- After every run the notebook prints one line `RESULT {...}` with all the numbers (accuracy, selected step, confusion matrix and calibration of the held-out domain, VRAM, time, trainable params). Keep the notebook saved with its outputs.
- Every finished run is saved immediately to **Google Drive** (`MyDrive/DeepLearningProject/M2_backup/<run>/`): `result.json`, the DomainBed logs (`results.jsonl`, `out.txt`, `err.txt`) (model weights are not saved: `SAVE_CHECKPOINTS = False` in section 4). Section 1 asks once for Drive access.
- In a later session the runs already on Drive are **skipped automatically**: just *Run all* again.

**Short sessions are fine.**
1. Set `SESSION_BUDGET_MIN` (section 4) to how long you can keep the tab open: no run is started that cannot finish in time.
2. Next session: *Run all* again. Finished runs are read from Drive and skipped; a run interrupted by a disconnect was not saved, so it is simply run again.
3. When all 4 runs are done, `M2_results.ipynb` reads them from Drive; add the result to the experiment log in `Milestone_2/README.md`.

**Runtime: A100 (or L4) GPU.** Set **Runtime -> Change runtime type -> A100 GPU** first.

## 1. Setup: Google Drive, clone DomainBed, install dependencies
Mounts Google Drive for saving every run. Following the course guideline, the pre-installed PyTorch of Colab is used and only the missing packages are installed: DINOv2 loads via `torch.hub`; CLIP needs `open_clip_torch`; `wilds` is imported unconditionally by DomainBed; `datasets` fetches PACS from Hugging Face.

In [1]:
import time
SESSION_T0 = time.time()          # the session budget (section 6) counts from here

# Every finished run is saved right away to Google Drive (course guideline: cloud sessions
# reset): MyDrive/DeepLearningProject/M2_backup/<run>/
# Each run folder holds result.json (the RESULT line) and the DomainBed logs (results.jsonl,
# out.txt, err.txt). Model weights only if SAVE_CHECKPOINTS = True (section 4).
# Finished runs found there are skipped automatically in later sessions.

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os
CODE_ROOT  = '/content'                                          # DomainBed clone
LOCAL_DATA = os.path.join(CODE_ROOT, 'data')                      # PACS
RUNS_LOCAL = os.path.join(CODE_ROOT, 'M2_runs')                   # scratch run dirs
from google.colab import drive
drive.mount('/content/drive')
BACKUP_DIR = '/content/drive/MyDrive/DeepLearningProject/M2_backup'
os.makedirs(RUNS_LOCAL, exist_ok=True); os.makedirs(BACKUP_DIR, exist_ok=True)
print('runs are saved to', BACKUP_DIR)

%cd $CODE_ROOT
# Always start from a PRISTINE clone so the patches below apply to clean source.
!rm -rf DomainBed
!git clone -q https://github.com/facebookresearch/DomainBed.git
!pip -q install open_clip_torch wilds datasets
%cd DomainBed
!git log --oneline -1

NVIDIA A100-SXM4-80GB, 81920 MiB
Mounted at /content/drive
runs are saved to /content/drive/MyDrive/DeepLearningProject/M2_backup
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
/content/DomainBed
b93c22a (HEAD -> main, origin/main, origin/HEAD) Merge pull request #156 from kiranchari/URM


## 2. Patch DomainBed (CLIP, LoRA, head LR, RAM cache, TF32, error-analysis stats)
Identical in all six run notebooks. Every patch asserts its anchor, so a DomainBed change fails loudly instead of silently.

**Three mutually-exclusive adaptation modes**, decided by hparams:
- `linear_probe=True` -> backbone frozen, only the head trains.
- `lora=True` -> backbone frozen, rank-8 LoRA on the attention projections (fused qkv + output proj) + head train; identical on DINOv2 and CLIP (CLIP via `torch.nn.utils.parametrize`, since `nn.MultiheadAttention` cannot be wrapped).
- both `False` -> **full fine-tuning**, whole backbone trains; the new head gets 10x the backbone LR (`head_lr_mult`).

The RAM cache is written into `datasets.py` (so the training subprocess uses it) and stores images at native resolution: results are identical to the stock loader, only the per-step JPEG decode is removed.

**Error analysis**: at every checkpoint, `train.py` also computes the confusion matrix, the expected calibration error (ECE, 15 bins), the NLL and the mean confidence on the held-out domain. They are reported only for the checkpoint chosen by training-domain validation; the test domain is never used for selection.

In [2]:
import pathlib

# --- 2a. hparams_registry.py: CLIP + linear_probe + LoRA + head LR + DINOv2 CLS-only
hp = pathlib.Path('domainbed/hparams_registry.py')
src = hp.read_text()
anchor = "    _hparam('vit_attn_tune', False, lambda r: False)\n"
additions = (
    "    _hparam('clip', False, lambda r: False)\n"
    "    _hparam('clip_model', 'ViT-B-16', lambda r: 'ViT-B-16')\n"
    "    _hparam('clip_pretrained', 'laion2b_s34b_b88k', lambda r: 'laion2b_s34b_b88k')\n"
    "    _hparam('linear_probe', False, lambda r: False)\n"
    "    _hparam('lora', False, lambda r: False)\n"
    "    _hparam('lora_r', 8, lambda r: 8)\n"
    "    _hparam('lora_alpha', 16, lambda r: 16)\n"
    "    _hparam('lora_dropout', 0.0, lambda r: 0.0)\n"
    "    _hparam('head_lr_mult', 1.0, lambda r: 1.0)\n"
    "    _hparam('dinov2_cls_only', False, lambda r: False)\n"
)
need = ['clip', 'clip_model', 'clip_pretrained', 'linear_probe', 'lora', 'lora_r',
        'lora_alpha', 'lora_dropout', 'head_lr_mult', 'dinov2_cls_only']
if "_hparam('head_lr_mult'," not in src:
    assert anchor in src, 'anchor line not found in hparams_registry.py'
    src = src.replace(anchor, anchor + additions, 1)
    hp.write_text(src)
check = hp.read_text()
for k in need:
    assert ("_hparam('%s'" % k) in check, 'MISSING hparam: %s' % k
print('OK: all hparams present.')

OK: all hparams present.


In [3]:
# --- 2b. networks.py: open_clip import, LoRA (Linear wrapper + parametrization), DinoV2 branches
nw = pathlib.Path('domainbed/networks.py')
src = nw.read_text()

# (i) lazy open_clip import after 'import timm'
if 'import open_clip' not in src:
    assert 'import timm\n' in src
    src = src.replace(
        'import timm\n',
        'import timm\n\ntry:\n    import open_clip\nexcept Exception:\n    open_clip = None\n',
        1)

# (ii) LoRA building blocks, inserted once before 'class ResNet'
lora_block = r'''
class LoRALinear(torch.nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank update (Milestone 2).

    y = W0 x + (alpha / r) * B (A x).  W0 is frozen; only A, B train.
    """
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r = int(r)
        self.scaling = float(alpha) / float(r)
        self.lora_dropout = nn.Dropout(dropout) if dropout and dropout > 0 else nn.Identity()
        in_f, out_f = base.in_features, base.out_features
        self.lora_A = nn.Parameter(torch.zeros(self.r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, self.r))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
        # lora_B stays zero -> adapter starts as a no-op (standard LoRA init)
    def forward(self, x):
        out = self.base(x)
        upd = self.lora_dropout(x) @ self.lora_A.t() @ self.lora_B.t()
        return out + self.scaling * upd


class LoRAWeightDelta(torch.nn.Module):
    """Parametrization W -> W + (alpha / r) * B A.

    Used for weights a module reads directly as tensors (nn.MultiheadAttention's
    in_proj_weight and out_proj.weight in open_clip), where wrapping the Linear
    is impossible. Same math and init as LoRALinear (without input dropout).
    """
    def __init__(self, out_f, in_f, r=8, alpha=16):
        super().__init__()
        self.r = int(r)
        self.scaling = float(alpha) / float(r)
        self.lora_A = nn.Parameter(torch.zeros(self.r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, self.r))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
    def forward(self, W):
        return W + self.scaling * (self.lora_B @ self.lora_A).to(W.dtype)


def _lora_parametrize(owner, name, r, alpha):
    import torch.nn.utils.parametrize as parametrize
    W = getattr(owner, name)
    W.requires_grad_(False)
    parametrize.register_parametrization(
        owner, name, LoRAWeightDelta(W.shape[0], W.shape[1], r, alpha))


LORA_LINEAR_TARGETS = ('qkv', 'proj')


def apply_lora_to_attention(module, r=8, alpha=16, dropout=0.0):
    """LoRA on the attention projections ONLY, identically for both backbones.

    - DINOv2 (torch.hub): Attention.qkv / Attention.proj are nn.Linear -> LoRALinear.
    - open_clip ViT: nn.MultiheadAttention reads in_proj_weight / out_proj.weight
      as raw tensors -> the same low-rank update is attached via parametrization.
    Either way: 2 adapted matrices per block (fused qkv 3d x d, output proj d x d),
    same rank -> same number of LoRA parameters per block on both backbones.
    Returns the number of adapted weight matrices."""
    n = 0
    for cname, cc in list(module.named_children()):
        if isinstance(getattr(cc, 'in_proj_weight', None), nn.Parameter) and \
                isinstance(getattr(cc, 'out_proj', None), nn.Linear):
            _lora_parametrize(cc, 'in_proj_weight', r, alpha)
            _lora_parametrize(cc.out_proj, 'weight', r, alpha)
            n += 2
        elif isinstance(cc, nn.Linear) and cname in LORA_LINEAR_TARGETS:
            setattr(module, cname, LoRALinear(cc, r, alpha, dropout))
            n += 1
        else:
            n += apply_lora_to_attention(cc, r, alpha, dropout)
    return n


'''
if 'class LoRAWeightDelta' not in src:
    assert 'class ResNet(torch.nn.Module):' in src, 'ResNet anchor not found'
    src = src.replace('class ResNet(torch.nn.Module):', lora_block + 'class ResNet(torch.nn.Module):', 1)

# (iii) DinoV2.__init__: linear_probe / lora / full-FT branch (full-FT = default)
dino_old = (
    "        self.dropout = nn.Dropout(hparams['vit_dropout'])\n"
    '\n'
    '        if hparams["vit_attn_tune"]:\n'
)
dino_new = (
    "        self.dropout = nn.Dropout(hparams['vit_dropout'])\n"
    '        self.n_lora = 0\n'
    '\n'
    '        if hparams.get("linear_probe", False):\n'
    '            for p in self.network.parameters():\n'
    '                p.requires_grad = False\n'
    '        elif hparams.get("lora", False):\n'
    '            for p in self.network.parameters():\n'
    '                p.requires_grad = False\n'
    '            self.n_lora = apply_lora_to_attention(\n'
    '                self.network, hparams.get("lora_r", 8),\n'
    '                hparams.get("lora_alpha", 16), hparams.get("lora_dropout", 0.0))\n'
    '            print("[LoRA] DINOv2: adapted %d attention weight matrices" % self.n_lora)\n'
    '        elif hparams["vit_attn_tune"]:\n'
)
if '[LoRA] DINOv2:' not in src:
    assert dino_old in src, 'DinoV2 base anchor not found -- inspect networks.py'
    src = src.replace(dino_old, dino_new, 1)

# (iv) optional DINOv2 CLS-only features (768-d) for the feature-dimension ablation
nout_old = "        self.n_outputs =  5 * 768\n"
nout_new = "        self.n_outputs = 768 if hparams.get('dinov2_cls_only', False) else 5 * 768\n"
fwd_old = "        x = self.network.get_intermediate_layers(x, n=4, return_class_token=True)\n"
fwd_new = (
    "        if self.hparams.get('dinov2_cls_only', False):\n"
    "            return self.dropout(self.network(x))  # final normed CLS token, 768-d\n"
    + fwd_old
)
if 'dinov2_cls_only' not in src:
    assert nout_old in src and fwd_old in src, 'DinoV2 n_outputs/forward anchor not found'
    src = src.replace(nout_old, nout_new, 1).replace(fwd_old, fwd_new, 1)

nw.write_text(src)
print('OK: LoRA blocks + DinoV2 branches written.')

OK: LoRA blocks + DinoV2 branches written.


In [4]:
# --- 2c. networks.py: add CLIP_ViT (with lora branch) + Featurizer selection
nw = pathlib.Path('domainbed/networks.py')
src = nw.read_text()

clip_cls = r'''
class CLIP_ViT(torch.nn.Module):
    """OpenCLIP ViT image tower as a DomainBed featurizer (Milestone 2).

    Supports linear_probe (frozen), lora (frozen + LoRA on attention), or full
    fine-tuning (default). DomainBed normalizes inputs with ImageNet mean/std,
    but CLIP expects its OWN mean/std -- we undo ImageNet norm and reapply CLIP's.
    """
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD  = (0.229, 0.224, 0.225)
    def __init__(self, input_shape, hparams):
        super(CLIP_ViT, self).__init__()
        if open_clip is None:
            raise RuntimeError("open_clip required: pip install open_clip_torch")
        nc = input_shape[0]
        if nc != 3:
            raise RuntimeError("Inputs must have 3 channels")
        model_name = hparams.get("clip_model", "ViT-B-16")
        pretrained = hparams.get("clip_pretrained", "laion2b_s34b_b88k")
        model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.network = model
        if hasattr(self.network, "transformer"):
            del self.network.transformer   # drop the text tower; image only
        n_out = getattr(getattr(model, "visual", None), "output_dim", None)
        if n_out is None:
            with torch.no_grad():
                n_out = int(model.encode_image(torch.zeros(1, *input_shape)).shape[-1])
        self.n_outputs = int(n_out)
        self.hparams = hparams
        self.dropout = nn.Dropout(hparams["vit_dropout"])
        clip_mean = clip_std = None
        try:
            cfg = open_clip.get_model_preprocess_cfg(self.network)
            clip_mean = cfg.get("mean"); clip_std = cfg.get("std")
        except Exception:
            pass
        if clip_mean is None or clip_std is None:
            clip_mean = (0.48145466, 0.4578275, 0.40821073)
            clip_std  = (0.26862954, 0.26130258, 0.27577711)
        self.register_buffer("imagenet_mean", torch.tensor(self.IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std",  torch.tensor(self.IMAGENET_STD).view(1, 3, 1, 1))
        self.register_buffer("clip_mean", torch.tensor(clip_mean).view(1, 3, 1, 1))
        self.register_buffer("clip_std",  torch.tensor(clip_std).view(1, 3, 1, 1))
        self._frozen = False
        self.n_lora = 0
        if hparams.get("linear_probe", False):
            for p in self.network.parameters():
                p.requires_grad = False
            self._frozen = True
        elif hparams.get("lora", False):
            for p in self.network.parameters():
                p.requires_grad = False
            self.n_lora = apply_lora_to_attention(
                self.network.visual, hparams.get("lora_r", 8),
                hparams.get("lora_alpha", 16), hparams.get("lora_dropout", 0.0))
            print("[LoRA] CLIP: adapted %d attention weight matrices" % self.n_lora)
        elif hparams.get("vit_attn_tune", False):
            for n, p in self.network.named_parameters():
                p.requires_grad = ("attn" in n)
        # else: full fine-tuning -- all params remain trainable.
    def _renorm(self, x):
        x = x * self.imagenet_std + self.imagenet_mean
        return (x - self.clip_mean) / self.clip_std
    def train(self, mode=True):
        super().train(mode)
        if self._frozen:
            self.network.eval()
        return self
    def forward(self, x):
        x = self._renorm(x)
        if self._frozen:
            with torch.no_grad():
                feats = self.network.encode_image(x)
            return self.dropout(feats.float())
        feats = self.network.encode_image(x)
        return self.dropout(feats.float())


'''
if 'class CLIP_ViT' not in src:
    assert 'class ResNet(torch.nn.Module):' in src, 'ResNet anchor not found'
    src = src.replace('class ResNet(torch.nn.Module):', clip_cls + 'class ResNet(torch.nn.Module):', 1)

# Featurizer: select CLIP before DinoV2 when hparams['clip'] is set
feat_old = (
    '        if hparams["vit"]:\n'
    '            if hparams["dinov2"]:\n'
)
feat_new = (
    '        if hparams["vit"]:\n'
    '            if hparams.get("clip", False):\n'
    '                return CLIP_ViT(input_shape, hparams)\n'
    '            if hparams["dinov2"]:\n'
)
if 'return CLIP_ViT(input_shape, hparams)' not in src:
    assert feat_old in src, 'Featurizer anchor not found'
    src = src.replace(feat_old, feat_new, 1)

nw.write_text(src)
print('OK: CLIP_ViT + Featurizer selection written.')

OK: CLIP_ViT + Featurizer selection written.


In [5]:
# --- 2d. algorithms.py: separate LR for the linear head (ERM only) -------------
# head_lr_mult=1.0 builds exactly the stock optimizer; >1 gives the freshly
# initialised head a larger LR than the pretrained backbone (full fine-tuning).
al = pathlib.Path('domainbed/algorithms.py')
src = al.read_text()
opt_old = (
    "        self.network = nn.Sequential(self.featurizer, self.classifier)\n"
    "        self.optimizer = torch.optim.Adam(\n"
    "            self.network.parameters(),\n"
    "            lr=self.hparams[\"lr\"],\n"
    "            weight_decay=self.hparams['weight_decay']\n"
    "        )\n"
)
opt_new = (
    "        self.network = nn.Sequential(self.featurizer, self.classifier)\n"
    "        head_lr = self.hparams[\"lr\"] * self.hparams.get('head_lr_mult', 1.0)\n"
    "        self.optimizer = torch.optim.Adam(\n"
    "            [{'params': self.featurizer.parameters()},\n"
    "             {'params': self.classifier.parameters(), 'lr': head_lr}],\n"
    "            lr=self.hparams[\"lr\"],\n"
    "            weight_decay=self.hparams['weight_decay']\n"
    "        )\n"
)
if 'head_lr_mult' not in src:
    start = src.index('class ERM(Algorithm):')
    end = src.index('\nclass ', start + 1)
    erm = src[start:end]
    assert opt_old in erm, 'ERM optimizer anchor not found'
    src = src[:start] + erm.replace(opt_old, opt_new, 1) + src[end:]
    al.write_text(src)
print('OK: ERM head_lr_mult wired.')

OK: ERM head_lr_mult wired.


In [6]:
# --- 2e. datasets.py: decode-once RAM cache (speed only, results unchanged) ----
# Written INTO DomainBed so the training subprocess actually uses it (a
# monkey-patch in this kernel would not reach `python -m domainbed.scripts.train`).
# Images are cached at NATIVE resolution as the exact RGB pixels the stock
# loader produces, so every transform (Resize / RandomResizedCrop / jitter)
# sees the identical PIL image -> identical results, minus the per-step decode.
ds_path = pathlib.Path('domainbed/datasets.py')
src = ds_path.read_text()
cache_cls = r'''
class CachedImageFolder(ImageFolder):
    """ImageFolder that decodes every image once into RAM (~1.5 GB for PACS)."""
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)
        import numpy as _np
        self._cache = []
        for path, _ in self.samples:
            with Image.open(path) as im:
                self._cache.append(_np.asarray(im.convert('RGB'), dtype=_np.uint8).copy())
        print('[cache] %s: %d images decoded to RAM'
              % (os.path.basename(root.rstrip('/')), len(self._cache)), flush=True)

    def __getitem__(self, index):
        img = Image.fromarray(self._cache[index])
        target = self.targets[index]
        if self.transform is not None:
            img = self.transform(img)
        if self.target_transform is not None:
            target = self.target_transform(target)
        return img, target


'''
cls_anchor = 'class MultipleEnvironmentImageFolder(MultipleDomainDataset):'
use_old = "            env_dataset = ImageFolder(path,\n                transform=env_transform)\n"
use_new = "            env_dataset = CachedImageFolder(path,\n                transform=env_transform)\n"
if 'class CachedImageFolder' not in src:
    assert cls_anchor in src and use_old in src, 'datasets.py anchors not found'
    src = src.replace(cls_anchor, cache_cls + cls_anchor, 1).replace(use_old, use_new, 1)
    ds_path.write_text(src)
print('OK: RAM cache written into datasets.py.')

OK: RAM cache written into datasets.py.


In [7]:
# --- 2f. train.py: enable TF32 matmuls (A100/L4) ----------------------------
# ~1.5-2x faster fp32 matmuls on Ampere+ GPUs; negligible accuracy effect for ViTs.
tr = pathlib.Path('domainbed/scripts/train.py')
src = tr.read_text()
tf_old = 'if __name__ == "__main__":\n'
tf_new = ('torch.backends.cuda.matmul.allow_tf32 = True\n'
          'torch.backends.cudnn.allow_tf32 = True\n\n' + tf_old)
if 'allow_tf32' not in src:
    assert src.count(tf_old) == 1, 'train.py __main__ anchor not found'
    src = src.replace(tf_old, tf_new, 1)
    tr.write_text(src)
print('OK: TF32 enabled in train.py.')

OK: TF32 enabled in train.py.


In [8]:
# --- 2g. train.py: confusion matrix + calibration on the held-out domain ------
# For error analysis (Milestone 2/3). Computed at every checkpoint on the test domain's
# 'in' split, in the same pass that computes its accuracy. It is only REPORTED for the
# checkpoint chosen by training-domain validation -- never used for model selection.
tr = pathlib.Path('domainbed/scripts/train.py')
src = tr.read_text()
fn_anchor = 'if __name__ == "__main__":\n'
fn_code = (
    "def m2_eval_stats(network, loader, device, n_bins=15):\n"
    "    # accuracy, confusion matrix, ECE, NLL and mean confidence on one loader\n"
    "    import torch.nn.functional as F\n"
    "    probs, ys = [], []\n"
    "    network.eval()\n"
    "    with torch.no_grad():\n"
    "        for x, y in loader:\n"
    "            probs.append(F.softmax(network.predict(x.to(device)).float(), dim=1).cpu())\n"
    "            ys.append(y.cpu())\n"
    "    network.train()\n"
    "    P, Y = torch.cat(probs), torch.cat(ys)\n"
    "    conf, pred = P.max(1)\n"
    "    correct = pred.eq(Y).float()\n"
    "    K = P.shape[1]\n"
    "    cm = torch.zeros(K, K, dtype=torch.long)\n"
    "    for t, p in zip(Y.tolist(), pred.tolist()):\n"
    "        cm[t, p] += 1\n"
    "    edges = torch.linspace(0, 1, n_bins + 1)\n"
    "    ece = 0.0\n"
    "    for lo, hi in zip(edges[:-1], edges[1:]):\n"
    "        m = (conf > lo) & (conf <= hi)\n"
    "        if m.any():\n"
    "            ece += m.float().mean().item() * abs(correct[m].mean().item() - conf[m].mean().item())\n"
    "    nll = F.nll_loss(torch.log(P.clamp_min(1e-12)), Y).item()\n"
    "    return {'acc': correct.mean().item(), 'cm': cm.tolist(), 'ece': ece, 'nll': nll,\n"
    "            'conf': conf.mean().item()}\n"
    "\n\n"
)
loop_old = (
    "            for name, loader, weights in evals:\n"
    "                acc = misc.accuracy(algorithm, loader, weights, device)\n"
    "                results[name+'_acc'] = acc\n"
)
loop_new = (
    "            m2_extra = {}\n"
    "            for name, loader, weights in evals:\n"
    "                if name == 'env{}_in'.format(args.test_envs[0]) and weights is None:\n"
    "                    st = m2_eval_stats(algorithm, loader, device)\n"
    "                    results[name+'_acc'] = st.pop('acc')\n"
    "                    m2_extra = {'test_' + k: v for k, v in st.items()}\n"
    "                else:\n"
    "                    acc = misc.accuracy(algorithm, loader, weights, device)\n"
    "                    results[name+'_acc'] = acc\n"
)
upd_old = "            results.update({\n                'hparams': hparams,\n"
upd_new = "            results.update(m2_extra)   # added after printing (lists do not print)\n" + upd_old
if 'm2_eval_stats' not in src:
    assert src.count(fn_anchor) == 1 and src.count(loop_old) == 1 and src.count(upd_old) == 1, \
        'train.py anchors not found'
    src = src.replace(fn_anchor, fn_code + fn_anchor, 1)
    src = src.replace(loop_old, loop_new, 1).replace(upd_old, upd_new, 1)
    tr.write_text(src)
print('OK: error-analysis stats (confusion matrix, ECE, NLL) wired into train.py.')

OK: error-analysis stats (confusion matrix, ECE, NLL) wired into train.py.


In [9]:
# --- 2h. everything compiles and every patch is present -----------------------
import py_compile
for f in ['networks.py', 'hparams_registry.py', 'algorithms.py', 'datasets.py', 'scripts/train.py']:
    py_compile.compile('domainbed/' + f, doraise=True)
chk = pathlib.Path('domainbed/networks.py').read_text()
for tok in ['class CLIP_ViT', 'return CLIP_ViT(input_shape, hparams)', 'class LoRALinear',
            'class LoRAWeightDelta', 'def apply_lora_to_attention', 'import open_clip',
            '[LoRA] DINOv2:', 'dinov2_cls_only']:
    assert tok in chk, 'missing in networks.py: ' + tok
assert 'head_lr_mult' in pathlib.Path('domainbed/algorithms.py').read_text()
assert 'CachedImageFolder(path' in pathlib.Path('domainbed/datasets.py').read_text()
tr_src = pathlib.Path('domainbed/scripts/train.py').read_text()
assert 'allow_tf32' in tr_src and 'm2_eval_stats' in tr_src and 'results.update(m2_extra)' in tr_src
print('OK: DomainBed fully patched and compiles.')

OK: DomainBed fully patched and compiles.


## 3. Download PACS from Hugging Face to local disk
Fetch `flwrlabs/pacs` and lay it out as DomainBed expects (`PACS/<domain>/<class>/img`). Class names come from the dataset features (never hardcoded). Every image is re-encoded to a valid RGB JPEG.

In [10]:
import os, shutil
from datasets import load_dataset

DATA_DIR = os.path.join(LOCAL_DATA, 'PACS')

def n_images(root):
    if not os.path.isdir(root):
        return 0
    return sum(len(files) for _, _, files in os.walk(root))

if n_images(DATA_DIR) == 9991:
    print('PACS already on local disk, skipping download.')
else:
    shutil.rmtree(DATA_DIR, ignore_errors=True)          # start clean
    ds = load_dataset('flwrlabs/pacs', split='train')    # 9991 images
    class_names = ds.features['label'].names             # authoritative class order
    print('classes:', class_names)
    counts = {}
    for i, ex in enumerate(ds):
        domain = ex['domain']                            # art_painting | cartoon | photo | sketch
        cls    = class_names[ex['label']]
        out_dir = os.path.join(DATA_DIR, domain, cls)
        os.makedirs(out_dir, exist_ok=True)
        idx = counts.get((domain, cls), 0); counts[(domain, cls)] = idx + 1
        ex['image'].convert('RGB').save(os.path.join(out_dir, f'{idx:05d}.jpg'), 'JPEG', quality=95)
        if i % 2000 == 0:
            print('written', i, flush=True)

print('PACS ready at', DATA_DIR)
for dom in sorted(os.listdir(DATA_DIR)):
    print(f'{dom:14s} {n_images(os.path.join(DATA_DIR, dom))} img')
assert n_images(DATA_DIR) == 9991

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

classes: ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
written 0
written 2000
written 4000
written 6000
written 8000
PACS ready at /content/data/PACS
art_painting   2048 img
cartoon        2344 img
photo          1670 img
sketch         3929 img


## 4. Configuration
`METHOD` / `BACKBONES_TO_RUN` fix what this notebook runs. `TRIAL_SEEDS`: DomainBed `--trial_seed` (data split) and `--seed` (training). Milestone 2 uses seed 0 only, the same split as Milestone 1, so every method is compared on the same data. Steps: linear probe 2000, LoRA and full FT 1500 (~25 epochs of PACS). The baseline is the Milestone 1 run (ERM ResNet-50, 5001 steps, seed 0). Check the "best step" column in section 7: if it is often the last checkpoint, more steps would help.

**Resuming:** runs already saved on Drive are skipped automatically.

In [11]:
# ============================================================================
METHOD = 'lp'
BACKBONES_TO_RUN = ['clip']
SESSION_BUDGET_MIN = 120   # stop launching new runs after this many minutes from session start
SAVE_CHECKPOINTS = False   # True = also save the fine-tuned weights (fp16, ~200 MB per run); not needed for Milestone 2
EST_RUN_MIN = 9           # first guess per run; replaced by the measured time after one run

# Runs saved on Drive are skipped automatically. Only if a run is missing from Drive (e.g. run
# from another account), paste its RESULT lines here (any extra text is ignored).
PREVIOUS_RESULTS = """

"""
# ============================================================================

TRIAL_SEEDS = [0]   # one seed = the Milestone 1 split; add 1, 2 later only if mean ± std is needed
METHODS = {
    # Milestone 1 setting: default ERM hparams (lr 5e-5), DomainBed's default 5001 steps for PACS
    'erm':    {'hp': {},                                  'steps': 5001},
    # lr = registry default (5e-5), 2000 steps: same as the earlier linear-probe runs
    'lp':     {'hp': {'linear_probe': True},             'steps': 2000},
    'lora':   {'hp': {'lora': True, 'lr': 5e-4},         'steps': 1500},
    'fullft': {'hp': {'lr': 1e-5, 'head_lr_mult': 10.0}, 'steps': 1500},
}
CLIP_HP = {'clip_model': 'ViT-B-16', 'clip_pretrained': 'laion2b_s34b_b88k'}
BACKBONES = {
    'resnet50':   {'vit': False, 'dinov2': False, 'clip': False, 'resnet50_augmix': False},
    'dinov2':     {'vit': True, 'dinov2': True,  'clip': False, 'resnet50_augmix': False},
    'clip':       {'vit': True, 'dinov2': False, 'clip': True,  'resnet50_augmix': False, **CLIP_HP},
    'dinov2_cls': {'vit': True, 'dinov2': True,  'clip': False, 'resnet50_augmix': False,
                   'dinov2_cls_only': True},
}

def run_name(method, bname, te, seed):
    return f'{method}_{bname}_env{te}_s{seed}'

def is_done(d):
    # ONLY the 'done' file marks completion: results.jsonl is appended at every
    # checkpoint, so it also exists for runs that crashed half-way.
    return os.path.exists(os.path.join(d, 'done'))

import json, re
import numpy as np

def parse_results(text):
    """Extract every `RESULT {...}` line from pasted notebook output (anything else is ignored).
    A run that appears twice keeps its last occurrence."""
    out = {}
    for m in re.finditer(r'RESULT (\{.*\})', text):
        r = json.loads(m.group(1))
        out[(r['method'], r['backbone'], r['seed'], r['te'])] = r
    return out

fmt = lambda v: f'{np.mean(v):.1f} ± {np.std(v):.1f}' if len(v) > 1 else (f'{v[0]:.1f}' if v else '-')

def table(results, seeds, header=True):
    """Markdown table: per-domain accuracy (mean ± std over seeds), average, cost."""
    rows = ['| Backbone | Method | A | C | P | S | Avg | Seeds | ECE (%) | Trainable params | Peak VRAM (GB) | s/step | Best step (median / last) |',
            '|---|---|---|---|---|---|---|---|---|---|---|---|---|'] if header else []
    combos = sorted({(r['backbone'], r['method']) for r in results.values()},
                    key=lambda bm: (['resnet50', 'dinov2', 'clip', 'dinov2_cls'].index(bm[0]),
                                    ['erm', 'lp', 'lora', 'fullft'].index(bm[1])))
    for bname, method in combos:
        res = {(s, te): r for (m, b, s, te), r in results.items() if m == method and b == bname}
        per_dom = [[res[(s, te)]['acc'] for s in seeds if (s, te) in res] for te in range(4)]
        full = [s for s in seeds if all((s, te) in res for te in range(4))]
        avg = [np.mean([res[(s, te)]['acc'] for te in range(4)]) for s in full]
        r = list(res.values())
        rows.append(f'| {bname} | {method} | ' + ' | '.join(fmt(v) for v in per_dom)
                    + f' | **{fmt(avg)}** | {len(full)}'
                    + f" | {fmt([100 * x['ece'] for x in r if 'ece' in x])} | {r[0]['trainable']:,d}"
                    + f" | {max(x['mem'] for x in r):.1f} | {np.mean([x['step_t'] for x in r]):.3f}"
                    + f" | {int(np.median([x['best_step'] for x in r]))} / {r[0]['last_step']} |")
    return '\n'.join(rows)

RESULTS = parse_results(PREVIOUS_RESULTS)
# runs already saved on Drive are loaded automatically
for d in sorted(os.listdir(BACKUP_DIR)):
    p = os.path.join(BACKUP_DIR, d, 'result.json')
    if os.path.exists(p):
        r = json.load(open(p))
        RESULTS[(r['method'], r['backbone'], r['seed'], r['te'])] = r
done_here = sorted(f"s{s}/env{te}" for (m, b, s, te) in RESULTS if m == METHOD and b in BACKBONES_TO_RUN)
print(f'{len(RESULTS)} runs found on Drive (all notebooks); {len(done_here)} of this notebook already done and skipped:',
      done_here)

0 runs found on Drive (all notebooks); 0 of this notebook already done and skipped: []


## 5. Smoke test (fail fast, ~1 min)
Builds the model once and checks that exactly the expected parameters train (linear probe: none of the backbone; LoRA: 24 adapted matrices, 442,368 params, gradients reach every adapter; ERM / full FT: everything).

In [12]:
import gc, torch
torch.backends.cuda.matmul.allow_tf32 = True
from domainbed import hparams_registry, algorithms

TRAINABLE = {}
for bname in BACKBONES_TO_RUN:
    hp = hparams_registry.default_hparams('ERM', 'PACS')
    hp.update(BACKBONES[bname]); hp.update(METHODS[METHOD]['hp'])
    alg = algorithms.ERM((3, 224, 224), 7, 3, hp).cuda().train()
    feat = alg.featurizer
    f_train = sum(p.numel() for p in feat.parameters() if p.requires_grad)
    f_total = sum(p.numel() for p in feat.parameters())
    TRAINABLE[bname] = sum(p.numel() for p in alg.parameters() if p.requires_grad)
    print(f'{METHOD} {bname}: trainable={TRAINABLE[bname]:,d} (backbone {f_train:,d} / {f_total:,d})')

    if METHOD == 'lp':
        assert f_train == 0, 'linear probe must not train the backbone'
    elif METHOD == 'lora':
        # 12 blocks x (fused qkv 2304x768 + proj 768x768), rank 8 -> 24 matrices,
        # 442,368 params, IDENTICAL on DINOv2 and CLIP.
        assert feat.n_lora == 24, feat.n_lora
        assert f_train == 442368, f_train
        x = torch.randn(2, 3, 224, 224, device='cuda')
        alg.predict(x).float().pow(2).mean().backward()
        missing = [n for n, p in feat.named_parameters() if 'lora_' in n and p.grad is None]
        assert not missing, 'LoRA params without gradient: %s' % missing[:5]
    elif METHOD == 'erm':
        assert f_train == f_total, 'ERM baseline should train everything'
    else:
        assert f_train == f_total, 'full-FT should train everything'
        lrs = [g['lr'] for g in alg.optimizer.param_groups]
        assert abs(lrs[1] - 10 * lrs[0]) < 1e-12, lrs
        print('optimizer LRs (backbone, head):', lrs)
    del alg, feat; gc.collect(); torch.cuda.empty_cache()
print('OK: model builds and trains the expected parameters.')

/content/DomainBed/domainbed/lib/misc.py:30: SyntaxWarning: invalid escape sequence '\d'
  ''' return proj_{B(h, \delta)}(adv_h), Euclidean projection to Euclidean ball'''


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

lp clip: trainable=3,591 (backbone 0 / 111,792,129)
OK: model builds and trains the expected parameters.


## 6. Launch the runs
- runs listed in `PREVIOUS_RESULTS` are skipped;
- a partial local dir is DELETED before rerunning (otherwise DomainBed would append to the old `results.jsonl`);
- a run counts only if DomainBed wrote its `done` file; then one `RESULT {...}` line is printed;
- no new run starts if it would not finish within `SESSION_BUDGET_MIN`;
- every finished run is saved to Drive at once (result and logs).

In [13]:
import sys, shutil, subprocess

from domainbed import model_selection          # DomainBed's own model selection code
from domainbed.lib.query import Q

def select(run_dir):
    """Test-domain accuracy chosen by DomainBed's OFFICIAL 'training-domain validation set'
    method (model_selection.IIDAccuracySelectionMethod, the one collect_results uses), plus
    the step and the cost of the selected checkpoint."""
    recs = [json.loads(l) for l in open(os.path.join(run_dir, 'results.jsonl'))]
    te = recs[0]['args']['test_envs'][0]
    official = model_selection.IIDAccuracySelectionMethod.run_acc(Q(recs))
    # the same rule, to know WHICH checkpoint was selected (its step, stats); must agree
    val = lambda r: np.mean([r[f'env{i}_out_acc'] for i in range(4) if i != te])
    best = max(recs, key=val)
    assert abs(best[f'env{te}_in_acc'] - official['test_acc']) < 1e-12, 'selection mismatch'
    step_t = np.mean([r['step_time'] for r in recs if r['step'] > 0] or [recs[-1]['step_time']])
    return {'acc': round(100 * official['test_acc'], 2), 'best_step': best['step'],
            'last_step': recs[-1]['step'], 'mem': round(max(r['mem_gb'] for r in recs), 2),
            'step_t': round(float(step_t), 4),
            # error analysis of the SELECTED checkpoint (held-out domain, 'in' split)
            **{k[5:]: (round(best[k], 4) if isinstance(best[k], float) else best[k])
               for k in ('test_cm', 'test_ece', 'test_nll', 'test_conf') if k in best}}

LOG_FILES = ('results.jsonl', 'out.txt', 'err.txt')

def backup_run(local_dir, name, result):
    """Save the RESULT, the logs and the fp16 checkpoint (if any) of a finished run to BACKUP_DIR."""
    dst = os.path.join(BACKUP_DIR, name)
    os.makedirs(dst, exist_ok=True)
    for f in LOG_FILES:
        if os.path.exists(os.path.join(local_dir, f)):
            shutil.copy2(os.path.join(local_dir, f), dst)
    ck = os.path.join(local_dir, 'model.pkl')
    if os.path.exists(ck):
        import torch
        d = torch.load(ck, map_location='cpu', weights_only=False)
        d['model_dict'] = {k: (v.half() if torch.is_floating_point(v) else v)
                           for k, v in d['model_dict'].items()}
        out = os.path.join(dst, name + '_model_fp16.pkl')
        torch.save(d, out)
        os.remove(ck)
        print(f'>>> checkpoint saved: {out} ({os.path.getsize(out) / 2**20:.0f} MB)', flush=True)
    with open(os.path.join(dst, 'result.json'), 'w') as f:   # written last = run fully saved
        json.dump(result, f)
    print(f'>>> run saved to {dst}', flush=True)

mcfg = METHODS[METHOD]
run_min = None
stopped = False
for seed in TRIAL_SEEDS:                     # seeds OUTER: seed 0 completes first
    for bname in BACKBONES_TO_RUN:
        hp = dict(BACKBONES[bname]); hp.update(mcfg['hp'])
        hp_json = json.dumps(hp)
        for te in [0, 1, 2, 3]:
            name = run_name(METHOD, bname, te, seed)
            local_dir = os.path.join(RUNS_LOCAL, name)
            if (METHOD, bname, seed, te) in RESULTS:
                print(f'--- {name} already done (from PREVIOUS_RESULTS), skipping ---', flush=True)
                continue
            elapsed = (time.time() - SESSION_T0) / 60
            need = run_min or EST_RUN_MIN
            if elapsed + need > SESSION_BUDGET_MIN:
                print(f'### session budget: {elapsed:.0f} min used, next run needs ~{need:.0f} '
                      f'(budget {SESSION_BUDGET_MIN}). Stopping cleanly.')
                stopped = True
                break
            shutil.rmtree(local_dir, ignore_errors=True)   # never append to a partial run
            print(f'\n=========== {name} | hparams={hp_json} ===========', flush=True)
            cmd = [
                sys.executable, '-m', 'domainbed.scripts.train',
                '--data_dir', LOCAL_DATA,
                '--algorithm', 'ERM',
                '--dataset', 'PACS',
                '--test_envs', str(te),
                '--steps', str(mcfg['steps']),
                '--holdout_fraction', '0.2',
                '--output_dir', local_dir,
                '--trial_seed', str(seed),
                '--seed', str(seed),
                '--hparams', hp_json,
            ]
            if not SAVE_CHECKPOINTS:
                cmd.append('--skip_model_save')
            t0 = time.time()
            proc = subprocess.Popen(cmd, cwd=os.path.join(CODE_ROOT, 'DomainBed'),
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end='', flush=True)
            proc.wait()
            if proc.returncode == 0 and is_done(local_dir):
                took = (time.time() - t0) / 60
                run_min = max(run_min or 0, took)      # measured, worst case so far
                r = {'method': METHOD, 'backbone': bname, 'seed': seed, 'te': te,
                     **select(local_dir), 'minutes': round(took, 1), 'trainable': TRAINABLE[bname]}
                RESULTS[(METHOD, bname, seed, te)] = r
                print('RESULT ' + json.dumps(r), flush=True)
                backup_run(local_dir, name, r)
            else:
                print(f'!!! {name} FAILED (exit code {proc.returncode}); no RESULT, will rerun next time')
        if stopped:
            break
    if stopped:
        break
print('\nSESSION FINISHED' + (' (budget reached, more runs left)' if stopped else ''))


=========== lp_clip_env0_s0 | hparams={"vit": true, "dinov2": false, "clip": true, "resnet50_augmix": false, "clip_model": "ViT-B-16", "clip_pretrained": "laion2b_s34b_b88k", "linear_probe": true} ===========
/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
Environment:
	Python: 3.13.15
	PyTorch: 2.11.0+cu130
	Torchvision: 0.26.0+cu130
	CUDA: 13.0
	CUDNN: 92700
	NumPy: 2.1.3
	PIL: 11.3.0
Args:
	algorithm: ERM
	checkpoint_freq: None
	data_dir: /content/data
	dataset: PACS
	holdout_fraction: 0.2
	hparams: {"vit": true, "dinov2": false, "clip": true, "resnet50_augmix": false, "clip_model": "ViT-B-16", "clip_pretrained": "laion2b_s34b_b88k", "linear_probe": true}
	hparams_seed: 0
	output_dir: /

## 7. All results of this notebook
Prints every `RESULT` line (earlier sessions + this one) and the per-domain table (mean ± std over seeds). Everything is also on Drive, where `M2_results.ipynb` reads it.

In [14]:
# All RESULT lines of this notebook so far (earlier sessions + this one). They are also on Drive.
mine = {k: r for k, r in RESULTS.items() if k[0] == METHOD and k[1] in BACKBONES_TO_RUN}
for k in sorted(mine):
    print('RESULT ' + json.dumps(mine[k]))
n_total = 4 * len(TRIAL_SEEDS) * len(BACKBONES_TO_RUN)
print(f'\n{len(mine)}/{n_total} runs finished.\n')
if mine:
    print(table(mine, TRIAL_SEEDS))

RESULT {"method": "lp", "backbone": "clip", "seed": 0, "te": 0, "acc": 96.77, "best_step": 1500, "last_step": 1999, "mem": 1.25, "step_t": 0.0905, "cm": [[301, 0, 2, 0, 4, 0, 0], [0, 184, 1, 0, 0, 0, 4], [1, 0, 224, 1, 2, 0, 0], [0, 0, 0, 143, 1, 2, 0], [0, 0, 0, 0, 162, 0, 0], [0, 0, 0, 0, 0, 240, 1], [7, 0, 2, 8, 11, 6, 332]], "ece": 0.2887, "nll": 0.472, "conf": 0.679, "minutes": 4.8, "trainable": 3591}
RESULT {"method": "lp", "backbone": "clip", "seed": 0, "te": 1, "acc": 98.45, "best_step": 1999, "last_step": 1999, "mem": 1.25, "step_t": 0.0929, "cm": [[319, 0, 0, 0, 0, 0, 0], [0, 353, 0, 0, 0, 0, 0], [0, 0, 276, 0, 0, 0, 0], [0, 0, 0, 112, 0, 0, 0], [11, 0, 9, 0, 242, 0, 0], [0, 0, 0, 0, 0, 231, 0], [6, 0, 0, 0, 0, 3, 314]], "ece": 0.1612, "nll": 0.2404, "conf": 0.8234, "minutes": 4.8, "trainable": 3591}
RESULT {"method": "lp", "backbone": "clip", "seed": 0, "te": 2, "acc": 99.4, "best_step": 1999, "last_step": 1999, "mem": 1.25, "step_t": 0.0902, "cm": [[149, 0, 0, 0, 0, 0, 0], 

## 8. Cross-check with DomainBed's official `collect_results`
Runs DomainBed's own results script on the runs finished **in this session** (for all runs, point `--input_dir` to the Drive backup folder, which holds every `results.jsonl`). Read the row under **`model selection method: training-domain validation set`**: its accuracies must match the `RESULT` lines. Ignore the oracle row (it uses the test domain). With one run per domain the `+/-` shown by the script is 0.0 (no variance can be computed from one run).

In [15]:
finished = [d for d in os.listdir(RUNS_LOCAL) if is_done(os.path.join(RUNS_LOCAL, d))]
print(len(finished), 'runs of this session:', sorted(finished))
if finished:
    !python -m domainbed.scripts.collect_results --input_dir="$RUNS_LOCAL"

4 runs of this session: ['lp_clip_env0_s0', 'lp_clip_env1_s0', 'lp_clip_env2_s0', 'lp_clip_env3_s0']
/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
Total records: 32

-------- Dataset: PACS, model selection method: training-domain validation set
Algorithm             A                     C                     P                     S                     Avg                  
ERM                   96.8 +/- 0.0          98.5 +/- 0.0          99.4 +/- 0.0          92.1 +/- 0.0          96.7                 

-------- Averages, model selection method: training-domain validation set
Algorithm                  PACS                       Avg                       
ERM                        96.7 